# 🤝 Module 11 Lab: Human-in-the-Loop Design & Enterprise Workflow Integration

**FinTech Corp · LoanBot v3.0 HITL System**

Trong lab này, bạn sẽ xây dựng toàn bộ HITL infrastructure:

| Section | Nội dung |
|---------|----------|
| 1 | HITL Trigger Engine — phân loại & escalate cases |
| 2 | ApprovalWorkflow — multi-level, SLA tracking, state machine |
| 3 | ExplanationGenerator — XAI cho human reviewers |
| 4 | NotificationHub — Slack/Email/SMS (mocked) |
| 5 | FeedbackCollector & DriftDetector — continuous improvement |
| 6 | End-to-End Simulation — 4 LoanBot customers |

**Không cần API key thật** — toàn bộ simulation với mock functions.

In [ ]:
# Setup
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
from statistics import mean
import uuid
import hashlib
import json
import random

print('✅ Module 11 Lab — LoanBot HITL System initialized')
print('📅', datetime.now().strftime('%Y-%m-%d %H:%M'))

---
## Section 1: HITL Trigger Engine

Xác định khi nào LoanBot cần escalate sang human review.

**4 triggers:**
- T1: Loan Amount > 500M VNĐ
- T2: Fraud signal > 0.3
- T3: Borderline credit score (560–680) hoặc AI confidence < 0.75
- T4: Regulatory exception flag

In [ ]:
class HITLTrigger(Enum):
    AMOUNT     = "T1_AMOUNT"
    FRAUD      = "T2_FRAUD"
    BORDERLINE = "T3_BORDERLINE"
    REGULATORY = "T4_REGULATORY"


@dataclass
class HITLDecision:
    requires_hitl: bool
    triggers: List[HITLTrigger]
    approval_level: int          # 1 = Loan Officer, 2 = Branch Mgr, 3 = Credit Committee
    sla_hours: int
    reason: str


class HITLTriggerEngine:
    """Evaluates loan data against HITL rules."""

    AMOUNT_L2   = 500_000_000   # 500M VNĐ → Level 2
    AMOUNT_L3   = 2_000_000_000 # 2 tỷ VNĐ → Level 3 (Credit Committee)
    FRAUD_THRESH = 0.30
    SCORE_LOW    = 560
    SCORE_HIGH   = 680
    CONF_THRESH  = 0.75

    def evaluate(self,
                 loan_amount_vnd: int,
                 credit_score: int,
                 fraud_probability: float,
                 ai_confidence: float,
                 regulatory_flag: bool = False) -> HITLDecision:

        triggers: List[HITLTrigger] = []
        reasons  = []

        # T2: Fraud check FIRST — highest priority
        if fraud_probability > self.FRAUD_THRESH:
            triggers.append(HITLTrigger.FRAUD)
            reasons.append(f"Fraud probability {fraud_probability:.0%} > 30%")

        # T1: Amount check
        if loan_amount_vnd > self.AMOUNT_L2:
            triggers.append(HITLTrigger.AMOUNT)
            reasons.append(f"Loan {loan_amount_vnd/1e6:.0f}M > 500M VNĐ (EU AI Act Art.14)")

        # T3: Borderline score or low confidence
        if self.SCORE_LOW <= credit_score <= self.SCORE_HIGH:
            triggers.append(HITLTrigger.BORDERLINE)
            reasons.append(f"Credit score {credit_score} in borderline zone ({self.SCORE_LOW}–{self.SCORE_HIGH})")
        elif ai_confidence < self.CONF_THRESH:
            triggers.append(HITLTrigger.BORDERLINE)
            reasons.append(f"AI confidence {ai_confidence:.0%} < 75%")

        # T4: Regulatory
        if regulatory_flag:
            triggers.append(HITLTrigger.REGULATORY)
            reasons.append("Regulatory exception flag")

        if not triggers:
            return HITLDecision(
                requires_hitl=False, triggers=[],
                approval_level=0, sla_hours=0,
                reason="Auto-process: no triggers"
            )

        # Determine approval level
        if HITLTrigger.FRAUD in triggers or loan_amount_vnd > self.AMOUNT_L3:
            level, sla = 3, 24
        elif HITLTrigger.AMOUNT in triggers and loan_amount_vnd > self.AMOUNT_L2:
            level, sla = 2, 4
        else:
            level, sla = 1, 8

        return HITLDecision(
            requires_hitl=True, triggers=triggers,
            approval_level=level, sla_hours=sla,
            reason=" | ".join(reasons)
        )


# ── Test với 4 LoanBot customers ──────────────────────────────────────────────
engine = HITLTriggerEngine()

customers = [
    dict(id="FC-2024-001", loan=200_000_000, score=720, fraud=0.02, conf=0.91, reg=False),
    dict(id="FC-2024-002", loan=200_000_000, score=580, fraud=0.15, conf=0.68, reg=False),
    dict(id="FC-2024-003", loan=50_000_000,  score=400, fraud=0.62, conf=0.95, reg=False),
    dict(id="FC-2024-004", loan=300_000_000, score=645, fraud=0.08, conf=0.61, reg=False),
]

print("\n━━━━ HITL Trigger Evaluation ━━━━")
for c in customers:
    d = engine.evaluate(c['loan'], c['score'], c['fraud'], c['conf'], c['reg'])
    status = '🚨 HITL' if d.requires_hitl else '✅ AUTO'
    level_name = ['', 'Loan Officer', 'Branch Manager', 'Credit Committee']
    print(f"\n{c['id']} | {status}")
    if d.requires_hitl:
        print(f"  Triggers : {[t.value for t in d.triggers]}")
        print(f"  Level    : {d.approval_level} — {level_name[d.approval_level]}")
        print(f"  SLA      : {d.sla_hours} hours")
    print(f"  Reason   : {d.reason}")

---
## Section 2: Approval Workflow — State Machine & SLA Tracking

In [ ]:
class ApprovalStatus(Enum):
    PENDING     = "PENDING"
    IN_REVIEW   = "IN_REVIEW"
    ESCALATED   = "ESCALATED"
    APPROVED    = "APPROVED"
    REJECTED    = "REJECTED"
    AI_OVERRIDE = "AI_OVERRIDE"


@dataclass
class AuditEvent:
    timestamp: str
    action: str
    actor: str
    details: str


@dataclass
class HITLCase:
    case_id: str = field(default_factory=lambda: f"HITL-{uuid.uuid4().hex[:6].upper()}")
    loan_id: str = ""
    loan_amount_vnd: int = 0
    trigger_types: List[str] = field(default_factory=list)
    ai_recommendation: str = ""   # "APPROVE" | "REJECT"
    ai_confidence: float = 0.0
    status: ApprovalStatus = ApprovalStatus.PENDING
    assigned_to: str = ""
    approval_level: int = 1
    sla_hours: int = 8
    created_at: datetime = field(default_factory=datetime.now)
    sla_deadline: Optional[datetime] = None
    decision: str = ""
    decision_reason: str = ""
    decided_at: Optional[datetime] = None
    audit_trail: List[AuditEvent] = field(default_factory=list)

    def __post_init__(self):
        self.sla_deadline = self.created_at + timedelta(hours=self.sla_hours)
        self._log("CREATED", "system", f"Loan {self.loan_id}, triggers: {self.trigger_types}")

    def assign(self, reviewer: str):
        self.assigned_to = reviewer
        self.status = ApprovalStatus.IN_REVIEW
        self._log("ASSIGNED", "system", f"→ {reviewer}")

    def decide(self, decision: str, reason: str, reviewer: str):
        assert decision in ("APPROVE", "REJECT"), "decision must be APPROVE or REJECT"
        assert len(reason) >= 10, "Reason too short (min 10 chars — EU AI Act Art.14)"
        self.decision = decision
        self.decision_reason = reason
        self.decided_at = datetime.now()
        is_override = decision != self.ai_recommendation
        if is_override:
            self.status = ApprovalStatus.AI_OVERRIDE
            self._log("AI_OVERRIDE", reviewer, f"AI={self.ai_recommendation} → Human={decision} | {reason}")
        else:
            self.status = (ApprovalStatus.APPROVED if decision == "APPROVE" else ApprovalStatus.REJECTED)
            self._log(decision, reviewer, reason)
        return is_override

    def escalate(self, reason: str = "SLA breach"):
        self.approval_level += 1
        self.sla_hours = {1: 8, 2: 4, 3: 24}.get(self.approval_level, 24)
        self.sla_deadline = datetime.now() + timedelta(hours=self.sla_hours)
        self.status = ApprovalStatus.ESCALATED
        self._log("ESCALATED", "system", f"→ Level {self.approval_level} | {reason}")

    def sla_remaining_pct(self) -> float:
        if not self.sla_deadline:
            return 100.0
        total = timedelta(hours=self.sla_hours).total_seconds()
        elapsed = (datetime.now() - self.created_at).total_seconds()
        return max(0.0, (1 - elapsed / total) * 100)

    def _log(self, action: str, actor: str, details: str):
        self.audit_trail.append(AuditEvent(
            timestamp=datetime.now().isoformat(timespec='seconds'),
            action=action, actor=actor, details=details
        ))

    def print_audit(self):
        print(f"\n{'━'*55}")
        print(f"Case: {self.case_id} | Loan: {self.loan_id}")
        print(f"Status: {self.status.value} | AI: {self.ai_recommendation} ({self.ai_confidence:.0%})")
        print(f"{'─'*55}")
        for e in self.audit_trail:
            print(f"  [{e.timestamp}] {e.action:<15} by {e.actor:<20} | {e.details}")


# ── Simulate FC-2024-002 HITL case ───────────────────────────────────────────
case002 = HITLCase(
    loan_id="FC-2024-002",
    loan_amount_vnd=200_000_000,
    trigger_types=["T3_BORDERLINE"],
    ai_recommendation="REJECT",
    ai_confidence=0.68,
    approval_level=1,
    sla_hours=8,
)
case002.assign("nguyen.thi.b@fintech.vn")
is_override = case002.decide(
    decision="APPROVE",
    reason="Soft factors: stable business 5 yrs, family collateral, strong character reference from branch.",
    reviewer="nguyen.thi.b@fintech.vn"
)
print(f"\n🔔 AI Override: {is_override}")
case002.print_audit()

---
## Section 3: ExplanationGenerator — XAI cho Human Reviewers

Tạo Explanation Card từ agent outputs để loan officer hiểu được quyết định AI.

In [ ]:
@dataclass
class FactorContribution:
    name: str
    value: str
    contribution: float  # positive = toward APPROVE, negative = toward REJECT
    weight: str          # HIGH / MEDIUM / LOW


class ExplanationGenerator:
    """Generates human-readable HITL explanations from agent outputs."""

    def generate(self,
                 loan_data: dict,
                 agent_outputs: dict) -> dict:

        credit = agent_outputs.get('CreditAgent', {})
        income = agent_outputs.get('IncomeAgent', {})
        fraud  = agent_outputs.get('FraudAgent',  {})
        risk   = agent_outputs.get('RiskAgent',   {})

        factors = self._compute_factors(credit, income, fraud, loan_data)
        hitl_reason = self._hitl_reason(credit, risk)
        summary = self._summary(factors, risk)
        attention = self._attention_points(factors)

        return {
            'loan_id':          loan_data.get('loan_id'),
            'applicant':        loan_data.get('applicant_name'),
            'verdict':          risk.get('recommendation', 'REVIEW'),
            'confidence':       risk.get('confidence', 0.0),
            'factors':          factors,
            'summary':          summary,
            'hitl_reason':      hitl_reason,
            'attention_points': attention,
        }

    def _compute_factors(self, credit, income, fraud, loan_data) -> List[FactorContribution]:
        factors = []
        score = credit.get('credit_score', 600)

        # Credit score factor
        score_contrib = (score - 600) / 400  # normalize roughly
        weight = 'HIGH' if abs(score_contrib) > 0.15 else 'MEDIUM'
        factors.append(FactorContribution(
            name=f"Credit Score ({score}/850)",
            value=str(score),
            contribution=round(score_contrib, 2),
            weight=weight
        ))

        # Income stability
        income_monthly = income.get('monthly_income_vnd', 0)
        income_std     = income.get('income_std_vnd', 0)
        is_self_emp    = credit.get('is_self_employed', False)
        if income_monthly > 0:
            cv = income_std / income_monthly  # coefficient of variation
            stability_contrib = -cv * 0.5 if is_self_emp else 0.1
            label = f"Thu nhập {income_monthly/1e6:.0f}M/tháng" + (" (tự kinh doanh)" if is_self_emp else "")
            factors.append(FactorContribution(
                name=label, value=f"{income_monthly/1e6:.0f}M VNĐ",
                contribution=round(stability_contrib, 2),
                weight='MEDIUM' if abs(stability_contrib) < 0.3 else 'HIGH'
            ))

        # DTI ratio
        dti = credit.get('dti_ratio', 0.0)
        if dti > 0:
            dti_contrib = 0.2 if dti < 0.35 else (-0.3 if dti > 0.45 else 0.0)
            factors.append(FactorContribution(
                name=f"DTI Ratio ({dti:.0%})",
                value=f"{dti:.0%}",
                contribution=round(dti_contrib, 2),
                weight='HIGH' if dti > 0.45 else 'MEDIUM'
            ))

        # Fraud signal
        fp = fraud.get('fraud_probability', 0.0)
        if fp > 0.1:
            factors.append(FactorContribution(
                name=f"Fraud Signal ({fp:.0%})",
                value=f"{fp:.2f}",
                contribution=round(-fp * 0.8, 2),
                weight='HIGH' if fp > 0.25 else 'MEDIUM'
            ))

        return sorted(factors, key=lambda f: abs(f.contribution), reverse=True)

    def _hitl_reason(self, credit, risk) -> str:
        reasons = []
        score = credit.get('credit_score', 0)
        conf  = risk.get('confidence', 1.0)
        if 560 <= score <= 680:
            reasons.append(f"Credit score {score} trong borderline zone")
        if conf < 0.75:
            reasons.append(f"AI confidence thấp ({conf:.0%})")
        if credit.get('is_self_employed'):
            reasons.append("Thu nhập tự kinh doanh cần expert assessment")
        return " | ".join(reasons) or "Regulatory requirement (EU AI Act Art.14)"

    def _summary(self, factors, risk) -> str:
        positives = [f.name for f in factors if f.contribution > 0.1]
        negatives = [f.name for f in factors if f.contribution < -0.1]
        verdict = risk.get('recommendation', 'REVIEW')
        conf    = risk.get('confidence', 0.0)
        return (f"AI recommend {verdict} (confidence {conf:.0%}). "
                f"Điểm tích cực: {', '.join(positives[:2]) or 'không rõ'}. "
                f"Điểm lo ngại: {', '.join(negatives[:2]) or 'không rõ'}.")

    def _attention_points(self, factors) -> List[str]:
        return [
            f"⚠️ {f.name}: contribution = {f.contribution:+.2f} [{f.weight}]"
            for f in factors if f.weight == 'HIGH'
        ]

    def print_card(self, explanation: dict):
        print(f"\n{'═'*58}")
        print(f"  EXPLANATION CARD — {explanation['loan_id']}")
        print(f"  Applicant : {explanation['applicant']}")
        print(f"  Verdict   : {explanation['verdict']} (Confidence: {explanation['confidence']:.0%})")
        print(f"{'─'*58}")
        print(f"  HITL Reason: {explanation['hitl_reason']}")
        print(f"{'─'*58}")
        print("  FACTOR CONTRIBUTIONS:")
        bar_width = 25
        for f in explanation['factors']:
            filled = int(abs(f.contribution) * bar_width)
            bar = ('█' * filled).ljust(bar_width)
            direction = '+' if f.contribution >= 0 else '-'
            color = '▶' if f.contribution >= 0 else '◀'
            print(f"  {color} {f.name:<35} {bar} {f.contribution:+.2f} [{f.weight}]")
        print(f"{'─'*58}")
        print(f"  SUMMARY: {explanation['summary']}")
        if explanation['attention_points']:
            print(f"{'─'*58}")
            print("  ATTENTION POINTS:")
            for a in explanation['attention_points']:
                print(f"    {a}")
        print(f"{'═'*58}")


# ── Test với FC-2024-004 ───────────────────────────────────────────────────
gen = ExplanationGenerator()

loan_004 = {
    'loan_id': 'FC-2024-004', 'applicant_name': 'Nguyễn Văn D',
    'loan_amount_vnd': 300_000_000
}
agent_outputs_004 = {
    'CreditAgent': {'credit_score': 645, 'is_self_employed': True, 'dti_ratio': 0.38},
    'IncomeAgent':  {'monthly_income_vnd': 22_000_000, 'income_std_vnd': 4_000_000},
    'FraudAgent':   {'fraud_probability': 0.08},
    'RiskAgent':    {'recommendation': 'REVIEW', 'confidence': 0.61},
}

expl = gen.generate(loan_004, agent_outputs_004)
gen.print_card(expl)

---
## Section 4: NotificationHub — Multi-Channel Notifications (Mocked)

In [ ]:
@dataclass
class NotificationRecord:
    notif_id: str
    channel: str
    recipient: str
    case_id: str
    sent_at: str
    content_preview: str


class NotificationHub:
    """Idempotent, multi-channel notification system for HITL cases."""

    def __init__(self):
        self._sent: Dict[str, NotificationRecord] = {}  # idempotency store
        self._log:  List[NotificationRecord]      = []

    def _notif_id(self, case_id: str, channel: str, recipient: str) -> str:
        return hashlib.md5(f"{case_id}-{channel}-{recipient}".encode()).hexdigest()[:12]

    def _send(self, channel: str, recipient: str, case: HITLCase,
              content: str) -> bool:
        nid = self._notif_id(case.case_id, channel, recipient)
        if nid in self._sent:
            print(f"    ⏩ SKIP (idempotent): {channel} → {recipient} already sent")
            return False
        # MOCK: in production, call real Slack/email/SMS API
        rec = NotificationRecord(
            notif_id=nid, channel=channel, recipient=recipient,
            case_id=case.case_id, sent_at=datetime.now().isoformat(timespec='seconds'),
            content_preview=content[:80]
        )
        self._sent[nid] = rec
        self._log.append(rec)
        print(f"    ✉️  SENT [{channel:6s}] → {recipient:<35} | {content[:60]}")
        return True

    def notify_new_case(self, case: HITLCase, reviewer_email: str,
                        manager_email: str):
        level_name = {1: 'Loan Officer', 2: 'Branch Manager', 3: 'Credit Committee'}
        slk_content = (f"🤝 HITL Case {case.case_id} | Loan {case.loan_id} "
                       f"{case.loan_amount_vnd/1e6:.0f}M VNĐ | "
                       f"AI={case.ai_recommendation} {case.ai_confidence:.0%} | "
                       f"SLA: {case.sla_hours}h | Review: https://hitl.fintech.vn/cases/{case.case_id}")
        email_content = (f"[HITL] New case {case.case_id} assigned to you. "
                         f"Loan: {case.loan_id}, {case.loan_amount_vnd/1e6:.0f}M VNĐ. "
                         f"SLA expires at {case.sla_deadline.strftime('%H:%M %d/%m/%Y')}")
        self._send('Slack', reviewer_email, case, slk_content)
        self._send('Email', reviewer_email, case, email_content)
        self._send('Email', manager_email,  case, f"[CC] HITL case {case.case_id} assigned to {reviewer_email}")

    def notify_sla_warning(self, case: HITLCase, reviewer_email: str):
        pct = case.sla_remaining_pct()
        sms_content = (f"⏰ SLA WARNING: HITL case {case.case_id} ({case.loan_id}) — "
                       f"{pct:.0f}% time remaining. Review NOW: https://hitl.fintech.vn/cases/{case.case_id}")
        self._send('SMS',   reviewer_email, case, sms_content)
        self._send('Slack', reviewer_email, case, sms_content)

    def notify_decision(self, case: HITLCase):
        is_override = case.status == ApprovalStatus.AI_OVERRIDE
        tag = "🔄 AI_OVERRIDE" if is_override else f"✅ {case.status.value}"
        content = (f"{tag}: {case.loan_id} → {case.decision} by {case.assigned_to} "
                   f"| Reason: {case.decision_reason[:60]}")
        self._send('Webhook', 'crm.salesforce.com', case, content)
        self._send('Webhook', 'corebanking.fintech.vn', case, content)

    def print_log(self):
        print(f"\n{'━'*65}")
        print(f" NOTIFICATION LOG ({len(self._log)} sent, {len(self._sent)} unique)")
        print(f"{'─'*65}")
        for r in self._log:
            print(f" {r.sent_at} [{r.channel:7s}] → {r.recipient:<30} | {r.content_preview}")


# ── Simulate notifications cho FC-2024-004 ────────────────────────────────
hub = NotificationHub()
case004 = HITLCase(
    loan_id="FC-2024-004", loan_amount_vnd=300_000_000,
    trigger_types=["T3_BORDERLINE"], ai_recommendation="REVIEW",
    ai_confidence=0.61, approval_level=1, sla_hours=8,
)

print("\n─── Step 1: New case notifications ───")
hub.notify_new_case(case004, reviewer_email="tran.van.c@fintech.vn", manager_email="mgr.branch3@fintech.vn")

print("\n─── Step 2: Retry (should be idempotent) ───")
hub.notify_new_case(case004, reviewer_email="tran.van.c@fintech.vn", manager_email="mgr.branch3@fintech.vn")

print("\n─── Step 3: SLA warning ───")
hub.notify_sla_warning(case004, reviewer_email="tran.van.c@fintech.vn")

print("\n─── Step 4: Decision made ───")
case004.assign("tran.van.c@fintech.vn")
case004.decide("APPROVE", "Self-employed 7 years, stable sector, collateral provided.", "tran.van.c@fintech.vn")
hub.notify_decision(case004)

hub.print_log()

---
## Section 5: FeedbackCollector & DriftDetector

In [ ]:
@dataclass
class OverrideRecord:
    case_id: str
    loan_id: str
    month: str                  # "2026-01"
    ai_recommendation: str
    human_decision: str
    reason: str
    segment: str                # "self_employed", "high_amount", "borderline", etc.
    loan_amount: int
    credit_score: int


class FeedbackCollector:
    """Collects HITL overrides and generates model improvement signals."""

    def __init__(self):
        self._overrides: List[OverrideRecord] = []
        self._monthly_override_rates: Dict[str, float] = {}  # {"2026-01": 0.03}
        self._monthly_hitl_counts:    Dict[str, int]   = {}

    def record_case(self, case: HITLCase, segment: str,
                    credit_score: int, month: str):
        self._monthly_hitl_counts[month] = self._monthly_hitl_counts.get(month, 0) + 1
        is_override = case.status == ApprovalStatus.AI_OVERRIDE
        if is_override and case.decision:
            self._overrides.append(OverrideRecord(
                case_id=case.case_id, loan_id=case.loan_id, month=month,
                ai_recommendation=case.ai_recommendation,
                human_decision=case.decision,
                reason=case.decision_reason, segment=segment,
                loan_amount=case.loan_amount_vnd, credit_score=credit_score,
            ))

    def update_override_rate(self, month: str, total_apps: int):
        n_overrides = sum(1 for o in self._overrides if o.month == month)
        self._monthly_override_rates[month] = n_overrides / max(total_apps, 1)

    def segment_analysis(self) -> Dict[str, Dict]:
        seg_data: Dict[str, Dict] = defaultdict(lambda: {'overrides': 0, 'reasons': []})
        for o in self._overrides:
            seg_data[o.segment]['overrides'] += 1
            seg_data[o.segment]['reasons'].append(o.reason)
        return dict(sorted(seg_data.items(), key=lambda x: x[1]['overrides'], reverse=True))

    def should_retrain(self) -> Tuple[bool, str]:
        rates = list(self._monthly_override_rates.values())
        if len(rates) < 4:
            return False, "Not enough history (need 4+ months)"
        current  = rates[-1]
        baseline = mean(rates[-4:-1])
        ratio    = current / baseline if baseline > 0 else 0
        if ratio > 1.5:
            return True, f"Override rate {current:.1%} > 1.5× baseline {baseline:.1%} ({ratio:.1f}×)"
        return False, f"Override rate {current:.1%} within normal range (baseline {baseline:.1%})"

    def training_dataset(self) -> List[Dict]:
        """Export structured dataset for model fine-tuning."""
        return [
            {
                'input': {'loan_id': o.loan_id, 'credit_score': o.credit_score,
                          'loan_amount': o.loan_amount, 'segment': o.segment},
                'ai_label':    o.ai_recommendation,
                'human_label': o.human_decision,
                'reason':      o.reason,
                'month':       o.month,
            }
            for o in self._overrides
        ]

    def print_report(self):
        print("\n" + "═"*60)
        print(" FEEDBACK & DRIFT ANALYSIS REPORT")
        print("═"*60)
        print(f"\n Total overrides recorded: {len(self._overrides)}")

        print("\n Monthly Override Rates:")
        for month, rate in sorted(self._monthly_override_rates.items()):
            bar = '▓' * int(rate * 200)
            print(f"  {month}: {rate:.1%}  {bar}")

        print("\n Override by Segment (model weakness analysis):")
        for seg, data in self.segment_analysis().items():
            print(f"  {seg:<20}: {data['overrides']} overrides")
            if data['reasons']:
                print(f"    → Top reason: {data['reasons'][0][:70]}")

        should, reason = self.should_retrain()
        print(f"\n Retrain Signal: {'🚨 YES' if should else '✅ No'}")
        print(f"  Reason: {reason}")

        dataset = self.training_dataset()
        print(f"\n Training Dataset: {len(dataset)} labeled examples ready for fine-tuning")
        print("═"*60)


# ── Simulate 6 months of override data ────────────────────────────────────
collector = FeedbackCollector()
monthly_data = [
    ('2026-01', 0.5, [('self_employed', 620, 280e6, 'REJECT', 'APPROVE', 'Stable 5yr business')]),
    ('2026-02', 0.5, []),
    ('2026-03', 0.5, [('borderline', 665, 350e6, 'REJECT', 'APPROVE', 'Good family collateral')]),
    ('2026-04', 1.0, [('self_employed', 590, 420e6, 'REJECT', 'APPROVE', 'COVID recovery, stable now'),
                      ('self_employed', 610, 300e6, 'REJECT', 'APPROVE', 'Industry voucher, strong cashflow')]),
    ('2026-05', 1.5, [('self_employed', 575, 500e6, 'REJECT', 'APPROVE', 'Pre-approved by branch manager'),
                      ('borderline', 640, 220e6, 'REJECT', 'APPROVE', 'Partner income not captured'),
                      ('self_employed', 600, 380e6, 'REJECT', 'APPROVE', 'Seasonal income pattern')]),
    ('2026-06', 2.5, [('self_employed', 580, 290e6, 'REJECT', 'APPROVE', 'Market expansion phase'),
                      ('self_employed', 595, 450e6, 'REJECT', 'APPROVE', 'Gov contract, guaranteed income'),
                      ('borderline', 650, 310e6, 'REJECT', 'APPROVE', 'New property purchase, future income up'),
                      ('self_employed', 570, 260e6, 'REJECT', 'APPROVE', 'Cash reserves not counted')]),
]

for month, override_rate_pct, cases in monthly_data:
    total_apps = 50_000
    for i, (seg, score, amt, ai_rec, hum_dec, reason) in enumerate(cases):
        case = HITLCase(
            loan_id=f"FC-{month[-2:]}-{i+1:03d}",
            loan_amount_vnd=int(amt), trigger_types=['T3_BORDERLINE'],
            ai_recommendation=ai_rec, ai_confidence=0.65,
            approval_level=1, sla_hours=8
        )
        case.assign("loan.officer@fintech.vn")
        case.decide(hum_dec, reason, "loan.officer@fintech.vn")
        collector.record_case(case, seg, score, month)
    collector.update_override_rate(month, total_apps // 333)  # ~150 HITL/month

collector.print_report()

print("\n Training Dataset Sample:")
for sample in collector.training_dataset()[:3]:
    print(f"  {json.dumps(sample, ensure_ascii=False, indent=2)[:200]}...")

---
## Section 6: End-to-End HITL Simulation — 4 LoanBot Customers

Chạy toàn bộ pipeline từ trigger detection → notification → human decision → feedback collection.

In [ ]:
def simulate_full_hitl_pipeline(customer: dict,
                                 agent_outputs: dict,
                                 human_decision: str,
                                 human_reason: str,
                                 reviewer: str,
                                 feedback_col: FeedbackCollector,
                                 notif_hub: NotificationHub):
    """End-to-end HITL pipeline for one loan application."""
    print(f"\n{'━'*65}")
    print(f" PROCESSING: {customer['loan_id']} — {customer['applicant_name']}")
    print(f"{'─'*65}")

    # Step 1: Trigger evaluation
    eng = HITLTriggerEngine()
    hitl = eng.evaluate(
        customer['loan_amount_vnd'],
        customer['credit_score'],
        customer['fraud_probability'],
        agent_outputs['RiskAgent']['confidence'],
        customer.get('regulatory_flag', False)
    )
    print(f"  [1] HITL Trigger: {hitl.requires_hitl} → {hitl.reason[:80]}")

    if not hitl.requires_hitl:
        print(f"  → AUTO decision: {agent_outputs['RiskAgent']['recommendation']}")
        return

    # Step 2: Create HITL case
    ai_rec = agent_outputs['RiskAgent']['recommendation']
    ai_conf = agent_outputs['RiskAgent']['confidence']
    case = HITLCase(
        loan_id=customer['loan_id'],
        loan_amount_vnd=customer['loan_amount_vnd'],
        trigger_types=[t.value for t in hitl.triggers],
        ai_recommendation=ai_rec, ai_confidence=ai_conf,
        approval_level=hitl.approval_level, sla_hours=hitl.sla_hours
    )
    print(f"  [2] Case created: {case.case_id} | Level {hitl.approval_level} | SLA {hitl.sla_hours}h")

    # Step 3: Generate explanation
    expl_gen = ExplanationGenerator()
    expl = expl_gen.generate(customer, agent_outputs)
    print(f"  [3] Explanation: {expl['summary'][:100]}")

    # Step 4: Notify reviewer
    mgr = "branch.mgr@fintech.vn" if hitl.approval_level >= 2 else "supervisor@fintech.vn"
    notif_hub.notify_new_case(case, reviewer, mgr)
    print(f"  [4] Notifications sent → {reviewer}")

    # Step 5: Human decision
    case.assign(reviewer)
    is_override = case.decide(human_decision, human_reason, reviewer)
    override_tag = "🔄 OVERRIDE" if is_override else "✅ Agree"
    print(f"  [5] Human decision: {human_decision} {override_tag} (AI was {ai_rec})")

    # Step 6: Post-decision notify
    notif_hub.notify_decision(case)

    # Step 7: Record feedback
    seg = 'self_employed' if customer.get('is_self_employed') else 'borderline'
    feedback_col.record_case(case, seg, customer['credit_score'], '2026-06')
    print(f"  [7] Feedback recorded → segment={seg}")

    case.print_audit()


# ── Setup ─────────────────────────────────────────────────────────────────
main_hub = NotificationHub()
main_col = FeedbackCollector()

# FC-2024-001: Good customer — auto approve
simulate_full_hitl_pipeline(
    customer={'loan_id':'FC-2024-001','applicant_name':'Trần Thị A','loan_amount_vnd':200_000_000,
              'credit_score':720,'fraud_probability':0.02,'is_self_employed':False},
    agent_outputs={'CreditAgent':{'credit_score':720,'dti_ratio':0.28,'is_self_employed':False},
                   'IncomeAgent':{'monthly_income_vnd':35_000_000,'income_std_vnd':1_000_000},
                   'FraudAgent':{'fraud_probability':0.02},
                   'RiskAgent':{'recommendation':'APPROVE','confidence':0.93}},
    human_decision='APPROVE', human_reason='N/A', reviewer='auto',
    feedback_col=main_col, notif_hub=main_hub
)

# FC-2024-002: Borderline — HITL, human overrides
simulate_full_hitl_pipeline(
    customer={'loan_id':'FC-2024-002','applicant_name':'Lê Văn B','loan_amount_vnd':200_000_000,
              'credit_score':580,'fraud_probability':0.15,'is_self_employed':False},
    agent_outputs={'CreditAgent':{'credit_score':580,'dti_ratio':0.42,'is_self_employed':False},
                   'IncomeAgent':{'monthly_income_vnd':18_000_000,'income_std_vnd':2_000_000},
                   'FraudAgent':{'fraud_probability':0.15},
                   'RiskAgent':{'recommendation':'REJECT','confidence':0.68}},
    human_decision='APPROVE',
    human_reason='Soft factors: 5yr business stability, strong character references, family collateral.',
    reviewer='nguyen.officer@fintech.vn',
    feedback_col=main_col, notif_hub=main_hub
)

# FC-2024-003: Blacklisted — HITL fraud (Level 3)
simulate_full_hitl_pipeline(
    customer={'loan_id':'FC-2024-003','applicant_name':'Phạm Thị C','loan_amount_vnd':50_000_000,
              'credit_score':400,'fraud_probability':0.62,'is_self_employed':False},
    agent_outputs={'CreditAgent':{'credit_score':400,'dti_ratio':0.65,'is_self_employed':False},
                   'IncomeAgent':{'monthly_income_vnd':8_000_000,'income_std_vnd':3_000_000},
                   'FraudAgent':{'fraud_probability':0.62},
                   'RiskAgent':{'recommendation':'REJECT','confidence':0.95}},
    human_decision='REJECT',
    human_reason='Confirmed fraud signals: blacklist match, suspicious application pattern.',
    reviewer='credit.committee@fintech.vn',
    feedback_col=main_col, notif_hub=main_hub
)

# FC-2024-004: Self-employed borderline — HITL, human overrides AI
simulate_full_hitl_pipeline(
    customer={'loan_id':'FC-2024-004','applicant_name':'Nguyễn Văn D','loan_amount_vnd':300_000_000,
              'credit_score':645,'fraud_probability':0.08,'is_self_employed':True},
    agent_outputs={'CreditAgent':{'credit_score':645,'dti_ratio':0.38,'is_self_employed':True},
                   'IncomeAgent':{'monthly_income_vnd':22_000_000,'income_std_vnd':4_000_000},
                   'FraudAgent':{'fraud_probability':0.08},
                   'RiskAgent':{'recommendation':'REJECT','confidence':0.61}},
    human_decision='APPROVE',
    human_reason='Self-employed 7 years in stable F&B sector. Property collateral 150% LTV. AI underweights sector stability.',
    reviewer='tran.manager@fintech.vn',
    feedback_col=main_col, notif_hub=main_hub
)

# ── Final summary ─────────────────────────────────────────────────────────
print("\n\n" + "═"*65)
print(" END-TO-END SIMULATION SUMMARY")
print("═"*65)
main_col.update_override_rate('2026-06', 150)
overrides = [r for r in main_col._overrides]
print(f"  Applications processed : 4")
print(f"  Auto-processed         : 1 (FC-2024-001)")
print(f"  HITL cases             : 3")
print(f"  AI Overrides           : {len(overrides)}")
print(f"  Override Rate          : {len(overrides)/3:.0%}")
print(f"  Notifications sent     : {len(main_hub._log)}")
dataset = main_col.training_dataset()
print(f"  Training examples      : {len(dataset)} labeled override records")
print("\n✅ Module 11 Lab complete — HITL System fully operational")

---
## Tổng kết: LoanBot HITL Architecture

```
                    ┌─────────────────────────────────┐
                    │        LoanBot v3.0 Pipeline     │
                    │  8 Agents → Orchestrator → Risk  │
                    └──────────────┬──────────────────┘
                                   │
                    ┌──────────────▼──────────────────┐
                    │      HITLTriggerEngine           │
                    │  T1:Amount T2:Fraud T3:Score T4 │
                    └──────────────┬──────────────────┘
                         │ HITL    │ Auto
              ┌──────────▼─┐   ┌──▼──────────┐
              │  HITLCase  │   │ Auto Approve │
              │ State Machine│   │  / Reject  │
              └──────┬─────┘   └─────────────┘
                     │
         ┌───────────┼───────────┐
         ▼           ▼           ▼
 ExplanationGen  NotifHub   Audit Trail
 (XAI Card)    (Slack/Email) (Immutable)
         │           │
         └─────┬─────┘
               ▼
        Human Reviewer
        (Loan Officer)
               │
         ┌─────▼────────────┐
         │ APPROVE / REJECT │
         │  (+ reason text) │
         └─────┬────────────┘
               │
    ┌──────────┴──────────────┐
    ▼                         ▼
FeedbackCollector        CRM/ERP/CoreBank
(DriftDetect + Retrain)   (via Webhook)
```

### Key Metrics LoanBot HITL
| Metric | Target | Formula |
|--------|--------|---------|
| HITL Rate | ~0.3% (150/50K apps/month) | HITL cases / total apps |
| Override Rate | < 5% của HITL cases | AI_OVERRIDE / HITL_total |
| SLA Breach Rate | < 2% | SLA_breached / HITL_total |
| Override-to-Training | 100% | All overrides captured as labeled data |
| Retrain Trigger | override rate > 1.5× baseline | Monthly monitoring |

### Next Steps cho LoanBot v3.1
1. **Integrate real Anthropic API** — thay mock agent_outputs bằng real multi-agent pipeline
2. **SHAP integration** — thay simplified factor_contributions bằng SHAP values thực
3. **Active Learning** — model tự chọn ambiguous cases để prioritize cho HITL
4. **Fine-tuning pipeline** — scheduled quarterly retraining từ override dataset
5. **A/B testing** — so sánh HITL UI designs về override rate và decision quality